# MyLLM - High-Speed GPU Training Notebook (Google Colab)
Optimized for GPU (T4 / A100 / L4) with fast non-overlapping chunk streaming, multi-threaded worker loading, and local SSD caching.

In [ ]:
# 1. Check GPU
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone or pull latest code from GitHub
import os
if not os.path.exists('/content/myllm'):
    !git clone https://github.com/aaravsuthar45-bit/myllm.git /content/myllm
%cd /content/myllm
!git pull

In [ ]:
# 3. Install requirements
%pip install -q tokenizers tqdm ftfy datasets

### 4. Locate or Copy Your Dataset
If your dataset (`training_data.jsonl` or `.bin`) is on Google Drive, mount Drive and copy it to Colab's fast local SSD (`/content/myllm/data`).

In [ ]:
from google.colab import drive
import shutil

# Mount Drive
drive.mount('/content/drive')

# Copy data from Drive to fast local Colab SSD (avoid slow Google Drive network I/O during training!)
# Change this path to where your training data is stored in your Google Drive:
DRIVE_DATA_SRC = '/content/drive/MyDrive/MY-LLM/data'

if os.path.exists(DRIVE_DATA_SRC):
    print(f"Copying data from {DRIVE_DATA_SRC} to local disk...")
    !rsync -av --progress "$DRIVE_DATA_SRC/" /content/myllm/data/
    print("Done copying data!")
else:
    print(f"[Info] {DRIVE_DATA_SRC} not found. Please verify the folder name in Drive, or upload training_data.jsonl to /content/myllm/data/ directly.")

### 5. Tokenize Newly Collected Data (If needed)
Converts `training_data.jsonl` into fast binary format (`fineweb_train.bin`).

In [ ]:
if os.path.exists('data/training_data.jsonl'):
    !python training/prepare_fineweb_pretrain.py \
        --input data/training_data.jsonl \
        --tokenizer data/tokenizer/tokenizer_combined.json \
        --output_dir data/pretraining \
        --val_fraction 0.05
else:
    print("data/training_data.jsonl not present, checking for pre-existing .bin files...")

### 6. Fast GPU Pretraining
Now using batch size 128, multi-worker loader, and non-overlapping token strides for massive speedup!

In [ ]:
# Pretraining execution with high-throughput parameters
!python training/pretrain.py \
    --batch_size 128 \
    --num_workers 2 \
    --epochs 3 \
    --learning_rate 5e-4

### 7. Test Generation / Inference

In [ ]:
!python generator.py --prompt "The future of artificial intelligence" --checkpoint checkpoints/pretrained_best.pt